In [139]:
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path
import os
import matplotlib.pyplot as plt
#for text pre-processing
import re, string
import nltk

nltk.download('stopwords')
nltk.download('omw-1.4')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
#for model-building
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.metrics import roc_curve, auc, roc_auc_score
# bag of words
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
#for word embedding
import gensim
from gensim.models import Word2Vec

[nltk_data] Downloading package stopwords to /home/javi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/javi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /home/javi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/javi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/javi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
#DEBBIE dataset.

#training set
training_set = [os.path.join("/home/javi/git/bsc/debbie/Classifier/classifier-development/trainning_set", f) for f in os.listdir("/home/javi/git/bsc/debbie/Classifier/classifier-development/trainning_set")]

#test_set 
test_set = [os.path.join("/home/javi/git/bsc/debbie/Classifier/classifier-development/test_set", f) for f in os.listdir("/home/javi/git/bsc/debbie/Classifier/classifier-development/test_set")]

print(training_set)
print(test_set)

In [ ]:
#making labels 
def make_labels(data):
    files = data
    file_list = []
    train_labels = [None] * (len(files))
    count = 0
    docID = 0
    for fil in files:
        file_list.append(fil)
        train_labels[docID] = 'non-relevant'
        filepathTokens = fil.split('/')
        lastToken = filepathTokens[len(filepathTokens) - 1]
        if fil.endswith("_clinical.txt"):
            train_labels[docID] = 'clinical'
            count = count + 1
        elif fil.endswith("_non-clinical.txt"):
            train_labels[docID] = 'non-clinical'
            count = count + 1
        docID = docID + 1
    combined_labels_files = dict(zip(train_labels, file_list))
    #print (train_labels)
    #print (file_list)
    print (combined_labels_files)
    return train_labels, file_list, combined_labels_files
    

#make labels for multiclass training and test set
training_set_labels, file_list, combined_training = make_labels(training_set)
test_set_labels, test_file_list, combined_testing  = make_labels(test_set)

In [ ]:
# to represent the distribution of the training set in a circular diagram


labels = ['non-relevant','non-clinical','clinical']

sizes = list(dict(Counter(training_set_labels)).values())
colours = {'non-relevant':'b','non-clinical':'g','clinical':'r'}
plt.pie(sizes,colors=[colours[key] for key in labels],autopct='%1.1f%%')
plt.legend(labels)
plt.title('Overview of the training set')
plt.show()

In [ ]:
# to represent the distribution of the test:


labels = ['non-relevant','non-clinical','clinical']

sizes = list(dict(Counter(test_set_labels)).values())
colours = {'non-relevant':'b','non-clinical':'g','clinical':'r'}
plt.pie(sizes,colors=[colours[key] for key in labels],autopct='%1.1f%%')
plt.legend(labels)
plt.title('Overview of the Test set')
plt.show()

In [ ]:
#convert to lowercase, strip and remove punctuations
def preprocess(text):
    text = text.lower() 
    text=text.strip()  
    text=re.compile('<.*?>').sub('', text) 
    text = re.compile('[%s]' % re.escape(string.punctuation)).sub(' ', text)  
    text = re.sub('\s+', ' ', text)  
    text = re.sub(r'\[[0-9]*\]',' ',text) 
    text=re.sub(r'[^\w\s]', '', str(text).lower().strip())
    text = re.sub(r'\d',' ',text) 
    text = re.sub(r'\s+',' ',text) 
    return text
 
# STOPWORD REMOVAL
def stopword(string):
    a= [i for i in string.split() if i not in stopwords.words('english')]
    return ' '.join(a)#LEMMATIZATION
# Initialize the lemmatizer
wl = WordNetLemmatizer()
 
# This is a helper function to map NTLK position tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN# Tokenize the sentence
def lemmatizer(string):
    word_pos_tags = nltk.pos_tag(word_tokenize(string)) # Get position tags
    a=[wl.lemmatize(tag[0], get_wordnet_pos(tag[1])) for idx, tag in enumerate(word_pos_tags)] # Map the position tag and lemmatize the word/token
    return " ".join(a)

def finalpreprocess(string):
    return lemmatizer(stopword(preprocess(string)))

In [ ]:
#load text of files 
X_train = [Path(file).read_text() for file in training_set]
X_test = [Path(file).read_text() for file in test_set]



In [130]:
X_train_preprocessed = [finalpreprocess(string) for string in X_train]
X_test_preprocessed = [finalpreprocess(string) for string in X_test]

In [145]:
#Word2Vec
# Word2Vec runs on tokenized sentences
X_train_tok= [nltk.word_tokenize(i) for i in X_train_preprocessed]  
X_test_tok= [nltk.word_tokenize(i) for i in X_test_preprocessed]

In [137]:
#vectorization using Bag-of-Words (with Tf-Idf )
#Tf-Idf
tfidf_vectorizer = TfidfVectorizer(use_idf=True)
X_train_vectors_tfidf = tfidf_vectorizer.fit_transform(X_train_preprocessed) 
X_test_vectors_tfidf = tfidf_vectorizer.transform(X_test_preprocessed)


In [140]:
lr_tfidf=LogisticRegression(solver = 'liblinear', C=10, penalty = 'l2')
lr_tfidf.fit(X_train_vectors_tfidf, training_set_labels)


#option I: SGD muticlass
sgd_clf = SGDClassifier(loss= 'log', penalty= 'l2').fit(X_train_vectors_tfidf, training_set_labels)


In [143]:
#Predict y value for test dataset
y_predict = sgd_clf.predict(X_test_vectors_tfidf)
y_prob = sgd_clf.predict_proba(X_test_vectors_tfidf)[:,1]
print(classification_report(test_set_labels,y_predict))
print('Confusion Matrix:',confusion_matrix(test_set_labels, y_predict))
 

              precision    recall  f1-score   support

    clinical       0.69      0.87      0.77       248
non-clinical       0.96      0.94      0.95       361
non-relevant       0.92      0.84      0.88       641

    accuracy                           0.88      1250
   macro avg       0.86      0.88      0.87      1250
weighted avg       0.89      0.88      0.88      1250

Confusion Matrix: [[216   6  26]
 [  3 338  20]
 [ 93   8 540]]


In [144]:
#Predict y value for test dataset
y_predict = lr_tfidf.predict(X_test_vectors_tfidf)
y_prob = lr_tfidf.predict_proba(X_test_vectors_tfidf)[:,1]
print(classification_report(test_set_labels,y_predict))
print('Confusion Matrix:',confusion_matrix(test_set_labels, y_predict))

              precision    recall  f1-score   support

    clinical       0.69      0.87      0.77       248
non-clinical       0.96      0.95      0.96       361
non-relevant       0.93      0.84      0.88       641

    accuracy                           0.88      1250
   macro avg       0.86      0.89      0.87      1250
weighted avg       0.89      0.88      0.88      1250

Confusion Matrix: [[215   6  27]
 [  2 343  16]
 [ 93   8 540]]


In [147]:
#building Word2Vec model
class MeanEmbeddingVectorizer(object):
    def __init__(self, word2vec):
        self.word2vec = word2vec
        # if a text is empty we should return a vector of zeros
        # with the same dimensionality as all the other vectors
        self.dim = len(next(iter(word2vec.values())))
    def fit(self, X, y):
        return self
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec[w] for w in words if w in self.word2vec]
                    or [np.zeros(self.dim)], axis=0)
            for words in X
        ])
w2v = dict(zip(model.wv.index2word, model.wv.syn0)) 
df['clean_text_tok']=[nltk.word_tokenize(i) for i in df['clean_text']]
model = Word2Vec(df['clean_text_tok'],min_count=1)     
modelw = MeanEmbeddingVectorizer(w2v)
# converting text to numerical data using Word2Vec
X_train_vectors_w2v = modelw.transform(X_train_tok)
X_val_vectors_w2v = modelw.transform(X_test_tok)

NameError: name 'model' is not defined